# RQ5, Part 1: Real Cross-Domain Point-Estimate Synthesis

Combines real effect sizes from RQ1 (averaged across all 3 real projects) and RQ4, using the established, correct averaging methodology.

**Requires the real RQ1 datasets (Camel/Hadoop, Kafka, Tika) and `data/rq4_real_sec_edgar_dataset_FINAL.csv`.**

In [6]:
!pip install -q pandas numpy statsmodels scipy || pip install -q pandas numpy statsmodels scipy --break-system-packages

In [7]:
"""
RQ5 - PART 1: Real Cross-Domain Point-Estimate Synthesis
================================================================================
Combines real effect sizes from RQ1 (averaged across all 3 real projects:
Camel/Hadoop, Kafka, Tika) and RQ4 (SEC EDGAR remediation), using the
established, correct methodology (real average, not max).
"""
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats


def rq1_project_f2(path, target):
    df = pd.read_csv(path)
    df["era_binary"] = (df["era"] == "ai_era").astype(int)
    full = smf.logit(f"{target} ~ (loc + cyclomatic_complexity + num_functions + num_files_changed) * era_binary", data=df).fit(disp=0)
    reduced = smf.logit(f"{target} ~ loc + cyclomatic_complexity + num_functions + num_files_changed + era_binary", data=df).fit(disp=0)
    r2f = 1 - (full.llf / full.llnull)
    r2r = 1 - (reduced.llf / reduced.llnull)
    return (r2f - r2r) / (1 - r2f)


if __name__ == "__main__":
    print("=== Real RQ1 effect sizes (3 projects) ===")
    f2_camel = rq1_project_f2("rq1_refined_with_real_issue_type.csv", "defect_prone_strict")
    f2_kafka = rq1_project_f2("kafka_real_mined_dataset.csv", "defect_prone")
    f2_tika = rq1_project_f2("tika_real_mined_dataset.csv", "defect_prone")
    print(f"  Camel/Hadoop: {f2_camel:.4f}")
    print(f"  Kafka: {f2_kafka:.4f}")
    print(f"  Tika: {f2_tika:.4f}")

    qa_avg = np.mean([f2_camel, f2_kafka, f2_tika])
    print(f"\nReal QA domain average (RQ1, 3 projects): {qa_avg:.4f}")

    print("\n=== Real RQ4 effect size (SEC EDGAR remediation) ===")
    df4 = pd.read_csv("rq4_real_sec_edgar_dataset_FINAL.csv", parse_dates=["disclosure_date", "remediation_date"])

    def consolidate_weakness(cat):
        cat = str(cat)
        if "Revenue" in cat: return "Revenue Recognition"
        if "ITGC" in cat: return "ITGC"
        if "Complex" in cat or "Warrant" in cat: return "Complex Transactions/Instruments"
        if "Control Environment" in cat or "Staffing" in cat or "Risk Assessment" in cat or "Segregation" in cat: return "Control Environment/Staffing"
        return "Other"

    def consolidate_industry(ind):
        ind = str(ind)
        if "Technology" in ind: return "Technology"
        if "Biotech" in ind or "Healthcare" in ind: return "Biotech/Healthcare"
        if "Manufacturing" in ind or "Industrial" in ind or "Aerospace" in ind or "Mining" in ind: return "Manufacturing/Industrial"
        if "SPAC" in ind: return "SPAC"
        if "Energy" in ind: return "Energy"
        if "Financial" in ind or "Insurance" in ind or "Real Estate" in ind: return "Financial/Real Estate"
        return "Media/Consumer/Other"

    df4["weakness_group"] = df4["weakness_category"].apply(consolidate_weakness)
    df4["industry_group"] = df4["industry"].apply(consolidate_industry)
    df4["disclosure_year"] = df4["disclosure_date"].dt.year

    full4 = smf.ols("remediation_days ~ C(weakness_group) + C(industry_group) + disclosure_year", data=df4).fit()
    reduced4 = smf.ols("remediation_days ~ C(weakness_group)", data=df4).fit()
    audit_f2 = (full4.rsquared - reduced4.rsquared) / (1 - full4.rsquared)
    print(f"Real audit domain effect size (RQ4): {audit_f2:.4f}")

    gap = abs(audit_f2 - qa_avg)
    print(f"\n=== REAL CROSS-DOMAIN COMPARISON ===")
    print(f"QA domain (avg of 3 real RQ1 projects): {qa_avg:.4f}")
    print(f"Audit domain (RQ4): {audit_f2:.4f}")
    print(f"Real absolute gap: {gap:.4f}")
    print(f"Pre-specified equivalence threshold: 0.10")
    print(f"Point estimate {'below' if gap < 0.10 else 'above'} threshold")
    print("\nNOTE: this point estimate alone is NOT the final, honest answer --")
    print("see Part 2 (bootstrap CI) for the real uncertainty around this gap.")


=== Real RQ1 effect sizes (3 projects) ===
  Camel/Hadoop: 0.0057
  Kafka: 0.0178
  Tika: 0.0038

Real QA domain average (RQ1, 3 projects): 0.0091

=== Real RQ4 effect size (SEC EDGAR remediation) ===
Real audit domain effect size (RQ4): 0.0624

=== REAL CROSS-DOMAIN COMPARISON ===
QA domain (avg of 3 real RQ1 projects): 0.0091
Audit domain (RQ4): 0.0624
Real absolute gap: 0.0534
Pre-specified equivalence threshold: 0.10
Point estimate below threshold

NOTE: this point estimate alone is NOT the final, honest answer --
see Part 2 (bootstrap CI) for the real uncertainty around this gap.
